# 🧊 Notebook 01 — Analyse Exploratoire des Données (EDA)
## Projet ThermoPath : Jumeau Numérique de Surveillance de la Chaîne du Froid

---

### 🎯 Objectif

Ce notebook constitue la **première étape analytique** du projet ThermoPath. Avant de construire un quelconque modèle de Machine Learning, il est impératif de :

1. **Valider la qualité** des données issues de la simulation MATLAB/Simulink (Software-in-the-Loop).
2. **Comprendre la dynamique physique** du caisson frigorifique : comment un choc mécanique sur une palette se traduit-il en dérive thermique ?
3. **Justifier le besoin** d'un feature engineering avancé (moyennes glissantes, vélocité thermique) pour alimenter un futur modèle de détection d'anomalies (Isolation Forest).

> ⚠️ **Ce notebook ne contient AUCUN Machine Learning.** Il s'agit d'une exploration pure des données brutes, guidée par la physique du système.

### 📦 Pipeline de données

Le dataset est chargé via le module `data_prep.py` du projet, qui effectue :
- Le filtrage par `batch_id`
- Le resampling à la minute (upsampling depuis des données horaires)
- L'interpolation temporelle de la température
- L'injection de chocs synthétiques via `features.py` (G-Force + dérive thermique)

---
## 2. 📥 Chargement et Inspection des Données

Nous utilisons le pipeline existant du projet (`data_prep.py` + `features.py`) pour obtenir un DataFrame réaliste avec température, g-force et indicateurs de chocs.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Ajout du répertoire racine au PATH pour importer les modules du projet
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_prep import load_and_resample
from src.features import inject_synthetic_faults

# ── Style graphique professionnel ───────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'lines.linewidth': 1.2,
    'figure.dpi': 120,
})

# Palette de couleurs du projet
COLORS = {
    'temp': '#1E88E5',       # Bleu froid
    'temp_roll': '#FF6F00',  # Orange vif
    'gforce': '#E53935',     # Rouge alerte
    'shock': '#FFD600',      # Jaune marqueur
    'bg': '#F5F5F5',
}

print("✅ Librairies importées et style configuré.")

In [ ]:
# ── Chargement via le pipeline du projet ────────────────────────────────
# 1) Charger et resampler les données brutes (batch001)
df_raw = load_and_resample(file_path="../data/raw/input_data.csv", batch_id="batch001")

# 2) Limiter à 5000 points pour une analyse lisible
df_raw = df_raw.iloc[:5000]

# 3) Injecter les chocs synthétiques (g_force + dérive thermique)
df = inject_synthetic_faults(df_raw, num_shocks=3, seed=42)

print(f"📐 Dimensions du DataFrame : {df.shape}")
print(f"📅 Période couverte : {df.index.min()} → {df.index.max()}")
print(f"⏱️  Pas de temps : {pd.infer_freq(df.index) or 'irrégulier'}")

In [ ]:
# ── Aperçu des premières lignes ─────────────────────────────────────────
df.head(10)

In [ ]:
# ── Informations structurelles ──────────────────────────────────────────
df.info()

In [ ]:
# ── Statistiques descriptives ───────────────────────────────────────────
df[['thermal_shipper_temp_reading', 'g_force', 'room_temp_reading',
    'room_humidity_reading']].describe().round(3)

In [ ]:
# ── Vérification des valeurs manquantes ─────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Manquantes': missing, '% du total': missing_pct})
print("🔍 Rapport des valeurs manquantes :")
print(missing_report[missing_report['Manquantes'] > 0] if missing.sum() > 0
      else "   ✅ Aucune valeur manquante détectée !")

### 📋 Observations préliminaires

- Le DataFrame contient les données d'un seul batch (`batch001`), resampleé à la **minute**.
- La colonne `thermal_shipper_temp_reading` représente la température du caisson frigorifique.
- La colonne `g_force` a été injectée synthétiquement : ~1.0 G au repos, avec des pics lors des chocs.
- La colonne `is_shock` est un indicateur binaire (1 = instant du choc mécanique).
- Aucune valeur manquante ne devrait subsister après le pipeline de préparation.

---
## 3. 📊 Analyse Univariée

Avant de croiser les variables, examinons la distribution individuelle de chaque signal physique.

### 3.1 Distribution de la Température

La température du caisson frigorifique devrait être centrée autour de valeurs basses (2-4°C pour un transport vaccinal standard). Les chocs injectés provoquent une dérive progressive vers le haut.

In [ ]:
# ── Distribution de la Température ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

sns.histplot(
    df['thermal_shipper_temp_reading'],
    bins=80, kde=True, color=COLORS['temp'],
    edgecolor='white', alpha=0.7, linewidth=0.5, ax=ax
)
ax.axvline(
    df['thermal_shipper_temp_reading'].median(),
    color='#FF6F00', linestyle='--', linewidth=2,
    label=f"Médiane = {df['thermal_shipper_temp_reading'].median():.2f} °C"
)
ax.set_title('Distribution de la Température du Caisson Frigorifique', fontweight='bold')
ax.set_xlabel('Température (°C)')
ax.set_ylabel('Fréquence')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**🔎 Interprétation :**
- La distribution est **majoritairement concentrée** autour de la température nominale du caisson.
- On observe une **queue à droite** (skew positif) : ce sont les dérives thermiques provoquées par les chocs mécaniques.
- Cette asymétrie est un premier indice que les anomalies thermiques ne sont pas des sauts brusques, mais des **dérives progressives**.

### 3.2 Distribution de la Force G (Accéléromètre)

Au repos, un accéléromètre mesure ~1.0 G (gravité terrestre). Les chocs mécaniques (nids-de-poule, collisions) produisent des pics bien au-delà de 3.0 G.

In [ ]:
# ── Distribution de la Force G (échelle logarithmique) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme linéaire
sns.histplot(
    df['g_force'], bins=100, color=COLORS['gforce'],
    edgecolor='white', alpha=0.7, ax=axes[0]
)
axes[0].set_title('Distribution G-Force (échelle linéaire)', fontweight='bold')
axes[0].set_xlabel('Force G')
axes[0].set_ylabel('Fréquence')

# Histogramme avec échelle log Y pour révéler les événements rares
sns.histplot(
    df['g_force'], bins=100, color=COLORS['gforce'],
    edgecolor='white', alpha=0.7, ax=axes[1]
)
axes[1].set_yscale('log')
axes[1].set_title('Distribution G-Force (échelle logarithmique)', fontweight='bold')
axes[1].set_xlabel('Force G')
axes[1].set_ylabel('Fréquence (log)')

# Annotation des seuils critiques
for ax in axes:
    ax.axvline(3.0, color='#FFD600', linestyle='--', linewidth=2,
               label='Seuil critique (3.0 G)')
    ax.legend()

plt.tight_layout()
plt.show()

# Statistiques des chocs
n_shocks = df['is_shock'].sum()
print(f"⚡ Nombre de chocs détectés : {n_shocks}")
print(f"📊 Valeur max de G-Force : {df['g_force'].max():.2f} G")
print(f"📊 Valeur moyenne (hors chocs) : {df[df['is_shock']==0]['g_force'].mean():.4f} G")

**🔎 Interprétation :**
- En **échelle linéaire**, les chocs sont quasiment invisibles : ils représentent une fraction infime des observations.
- L'**échelle logarithmique** révèle ces **événements rares mais critiques** au-delà de 3.0 G.
- C'est exactement ce type de déséquilibre (99.9% normal, 0.1% anomalie) qui rend les méthodes classiques de seuillage inadaptées et justifie l'utilisation d'un algorithme de détection d'anomalies comme l'**Isolation Forest**.

---
## 4. 🌡️ La Physique du Caisson : Visualisation Temporelle

> C'est **l'étape clé** de ce notebook. Nous allons superposer les signaux de G-Force et de Température pour observer visuellement le **couplage physique** entre un choc mécanique et la réponse thermique du caisson.

In [ ]:
# ── Calcul de la Moyenne Glissante (Rolling Mean) ───────────────────────
WINDOW = 5  # Fenêtre de 5 minutes
df['temp_rolling_mean'] = df['thermal_shipper_temp_reading'].rolling(
    window=WINDOW, center=False
).mean()

print(f"✅ Moyenne glissante calculée (fenêtre = {WINDOW} min)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GRAPHIQUE PRINCIPAL : Superposition G-Force / Température
# ═══════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(16, 10), sharex=True,
    gridspec_kw={'height_ratios': [1, 1.3], 'hspace': 0.08}
)

# ── Graphique du HAUT : Force G ─────────────────────────────────────────
ax1.plot(df.index, df['g_force'], color=COLORS['gforce'],
         alpha=0.8, linewidth=0.8, label='G-Force instantanée')
ax1.axhline(y=3.0, color='#FFD600', linestyle='--', linewidth=1.5,
            label='Seuil critique (3.0 G)', alpha=0.8)

# Marquage des instants de choc
shock_idx = df[df['is_shock'] == 1].index
for idx in shock_idx:
    ax1.axvline(x=idx, color=COLORS['shock'], alpha=0.3, linewidth=8)

ax1.set_ylabel('Force G', fontweight='bold', fontsize=13)
ax1.set_title('📡 Signal Accéléromètre & Réponse Thermique du Caisson',
              fontsize=15, fontweight='bold', pad=15)
ax1.legend(loc='upper right', fontsize=10)
ax1.set_ylim(0.7, df['g_force'].max() * 1.15)

# ── Graphique du BAS : Température + Rolling Mean ───────────────────────
ax2.plot(df.index, df['thermal_shipper_temp_reading'],
         color=COLORS['temp'], alpha=0.5, linewidth=0.7,
         label='Température brute')
ax2.plot(df.index, df['temp_rolling_mean'],
         color=COLORS['temp_roll'], linewidth=2.2, alpha=0.9,
         label=f'Moyenne glissante ({WINDOW} min)')

# Marquage des instants de choc (bandes verticales)
for idx in shock_idx:
    ax2.axvline(x=idx, color=COLORS['shock'], alpha=0.3, linewidth=8,
                label='Choc mécanique' if idx == shock_idx[0] else None)

ax2.set_ylabel('Température (°C)', fontweight='bold', fontsize=13)
ax2.set_xlabel('Temps', fontweight='bold', fontsize=13)
ax2.legend(loc='upper left', fontsize=10)

# Rotation des labels de l'axe X
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 🧠 Analyse Physique : L'Inertie Thermique

> **C'est le point fondamental de tout le projet ThermoPath.**

En observant le graphique ci-dessus, on constate un phénomène physique crucial : **la température ne réagit PAS instantanément au choc mécanique.**

Lorsqu'un choc survient (pic de G-Force visible en haut), la réponse thermique du caisson (graphique du bas) ne montre **aucun saut brutal**. Au lieu de cela, on observe une **dérive progressive** : la température commence à augmenter lentement, minute après minute, sur une fenêtre de ~60 minutes après l'impact.

Ce comportement est cohérent avec la **physique de la conduction thermique** :
- Le choc endommage l'isolation du caisson (joints, panneaux, scellés).
- L'air chaud extérieur s'infiltre progressivement dans le caisson.
- La masse thermique interne (vaccins, coolants) absorbe cette chaleur avec une **constante de temps** non négligeable.

#### ⚠️ Pourquoi une règle `if temp > seuil` est DANGEREUSE

Une approche naïve consisterait à déclencher une alerte dès que la température dépasse un seuil fixe (ex : 8°C). **Cette approche est fondamentalement défaillante** pour deux raisons :

1. **Détection trop tardive** : quand la température franchit le seuil, le dommage est déjà fait. Les vaccins ont été exposés à une température hors-norme pendant de longues minutes.
2. **Pas de causalité** : le seuil ne capture pas la *cause* (le choc mécanique), seulement la *conséquence* (la hausse de température). Il ne peut pas prédire.

#### ✅ Ce que la Moyenne Glissante révèle

La courbe orange (moyenne glissante sur 5 minutes) lisse le bruit haute fréquence et fait **émerger la tendance** de dérive thermique. C'est précisément ce type de feature (rolling mean, rolling std, vélocité thermique) qui alimentera notre **Isolation Forest** dans le notebook suivant.

> **Conclusion** : Un modèle prédictif capable de corréler un pic de G-Force avec une dérive thermique naissante permettrait de déclencher une alerte **avant** que la température ne devienne critique. C'est la valeur ajoutée du Machine Learning dans ce contexte industriel.

---
## 5. 🔬 Zoom sur un Choc Individuel

Pour mieux visualiser la dynamique post-choc, isolons une fenêtre temporelle autour du premier choc détecté.

In [ ]:
# ── Zoom sur le premier choc ────────────────────────────────────────────
if len(shock_idx) > 0:
    first_shock = shock_idx[0]
    # Fenêtre : 10 min avant → 80 min après le choc
    start = first_shock - pd.Timedelta(minutes=10)
    end = first_shock + pd.Timedelta(minutes=80)
    zoom = df.loc[start:end]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 8), sharex=True,
        gridspec_kw={'height_ratios': [0.8, 1.2], 'hspace': 0.08}
    )

    # G-Force
    ax1.plot(zoom.index, zoom['g_force'], color=COLORS['gforce'],
             linewidth=1.5, marker='o', markersize=2)
    ax1.axvline(x=first_shock, color=COLORS['shock'], linewidth=3, alpha=0.5,
                label=f'Choc @ {first_shock.strftime("%H:%M")}')
    ax1.set_ylabel('G-Force', fontweight='bold')
    ax1.set_title(f'🔬 Zoom — Choc du {first_shock.strftime("%d/%m/%Y à %H:%M")}',
                  fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right')

    # Température
    ax2.plot(zoom.index, zoom['thermal_shipper_temp_reading'],
             color=COLORS['temp'], alpha=0.6, linewidth=1, label='Temp. brute')
    ax2.plot(zoom.index, zoom['temp_rolling_mean'],
             color=COLORS['temp_roll'], linewidth=2.5, label=f'Moyenne glissante ({WINDOW} min)')
    ax2.axvline(x=first_shock, color=COLORS['shock'], linewidth=3, alpha=0.5)

    # Annotation de la dérive
    temp_at_shock = zoom.loc[first_shock, 'thermal_shipper_temp_reading']
    temp_end = zoom['thermal_shipper_temp_reading'].iloc[-1]
    delta = temp_end - temp_at_shock
    ax2.annotate(
        f'Δ = +{delta:.2f}°C\n(dérive post-choc)',
        xy=(zoom.index[-20], temp_end),
        fontsize=11, fontweight='bold', color='#D84315',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF3E0', edgecolor='#FF6F00')
    )

    ax2.set_ylabel('Température (°C)', fontweight='bold')
    ax2.set_xlabel('Temps', fontweight='bold')
    ax2.legend(loc='upper left')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    print(f"📊 Température au moment du choc : {temp_at_shock:.2f} °C")
    print(f"📊 Température 80 min après :      {temp_end:.2f} °C")
    print(f"📊 Dérive totale :                  +{delta:.2f} °C")
else:
    print("⚠️ Aucun choc détecté dans les données.")

---
## 6. 🔗 Matrice de Corrélation

Examinons les relations linéaires entre les variables numériques pour identifier les couplages potentiels.

In [ ]:
# ── Matrice de corrélation ──────────────────────────────────────────────
numeric_cols = ['thermal_shipper_temp_reading', 'g_force', 'is_shock',
                'room_temp_reading', 'room_humidity_reading']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.3f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=1, linecolor='white',
    cbar_kws={'shrink': 0.8, 'label': 'Corrélation de Pearson'},
    ax=ax
)
ax.set_title('Matrice de Corrélation des Variables Physiques',
             fontweight='bold', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

**🔎 Interprétation :**
- La corrélation entre `g_force` et `thermal_shipper_temp_reading` est probablement **faible** en valeur brute, car le lien est **temporellement décalé** (la température ne réagit que progressivement).
- C'est précisément cette **non-linéarité temporelle** qui rend nécessaire un feature engineering basé sur des **fenêtres glissantes** pour capturer la dynamique causale.

---
## 7. ✅ Conclusion et Prochaines Étapes

### Ce que nous avons appris

| Observation | Implication |
|---|---|
| La température dérive **progressivement** après un choc | Une règle de seuil simple est insuffisante |
| Les chocs mécaniques sont des **événements rares** | Le dataset est fortement déséquilibré → Isolation Forest |
| La corrélation brute G-Force/Temp est faible | Le lien est **temporel**, pas instantané → Rolling Features |
| La moyenne glissante lisse le bruit et révèle la tendance | Les features glissantes sont essentielles pour le modèle |

### Prochaines étapes (Notebook 02)

1. **Feature Engineering** : Calcul des rolling mean, rolling std, et vélocité thermique (`temp_velocity`) via le module `features.py`.
2. **Entraînement** d'un modèle **Isolation Forest** sur les features enrichies.
3. **Évaluation** de la capacité du modèle à détecter les anomalies thermiques causées par les chocs mécaniques.

> 🚀 **Le but ultime** : un système capable de déclencher une alerte dès qu'un choc mécanique est détecté, **avant** que la température ne dépasse le seuil critique — protégeant ainsi l'intégrité de la chaîne du froid.